# Run the Health Information Assistant on Colab

The demo needs a CUDA GPU: the INT8/INT4 checkpoints are BitsAndBytes
quantized, and BitsAndBytes only runs on CUDA. This notebook launches the
app from `webapp/` on a Colab GPU runtime and gives you a public share link.

## Before you start

1. **Runtime > Change runtime type > T4 GPU** (free tier is enough for the
   INT4 configurations).
2. Add your Hugging Face token as a Colab secret named `HF_TOKEN`
   (key icon in the left sidebar, then enable *Notebook access*).
3. Accept the licence on all three gated model pages with the same account:
   - <https://huggingface.co/google/gemma-4-E4B-it>
   - <https://huggingface.co/google/gemma-4-12B-it>
   - <https://huggingface.co/google/medgemma-1.5-4b-it>

Then run the cells in order.

**Research demo only.** It does not provide medical diagnosis and does not
replace a qualified healthcare professional.

## 1. Check the GPU

Stop here if this reports no GPU: change the runtime type first.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No CUDA GPU. Use Runtime > Change runtime type > T4 GPU, then rerun."
    )

props = torch.cuda.get_device_properties(0)
total_gib = props.total_memory / 2**30
print(f"GPU            : {props.name}")
print(f"VRAM           : {total_gib:.2f} GiB")
print(f"Compute cap.   : {props.major}.{props.minor}")
print(f"PyTorch        : {torch.__version__}")

# Peak VRAM measured by the benchmark, smallest first.
measured = [
    ("MedGemma 1.5 4B - INT4", 4.30),
    ("MedGemma 1.5 4B - INT8", 5.95),
    ("MedGemma 1.5 4B - Baseline BF16", 9.30),
    ("Gemma 4 12B - INT4", 9.85),
    ("Gemma 4 E4B - INT4", 10.80),
    ("Gemma 4 E4B - INT8", 12.80),
    ("Gemma 4 12B - INT8", 14.79),
    ("Gemma 4 E4B - Baseline BF16", 16.92),
    ("Gemma 4 12B - Baseline BF16", 24.98),
]
# Leave headroom for the KV cache and activations.
budget = total_gib - 1.5
print("\nConfigurations that should fit here:")
for label, vram in measured:
    print(f"  {'yes' if vram <= budget else 'no ':<4} {label:<32} {vram:>6.2f} GiB")
if props.major < 7:
    print("\nWarning: bitsandbytes targets compute capability 7.5+; this GPU is older.")

## 2. Get the code

If the repository is private, replace the URL with a token URL
(`https://<github-username>:<github-PAT>@github.com/peempat/medpubcodex.git`)
or upload the `webapp/` folder to `/content/medpubcodex/` by hand.

In [ ]:
REPO_URL = "https://github.com/peempat/medpubcodex.git"
BRANCH = "feature/webapp"
TARGET = "/content/medpubcodex"

import subprocess
from pathlib import Path

if Path(TARGET).exists():
    print("Already cloned; pulling the latest commit.")
    subprocess.run(["git", "-C", TARGET, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", TARGET, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", TARGET, "pull", "origin", BRANCH], check=True)
else:
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, TARGET],
        check=True,
    )

print("\nwebapp/ contents:")
for path in sorted(Path(TARGET, "webapp").iterdir()):
    print("  ", path.name)

## 3. Install dependencies

Colab already ships a CUDA build of PyTorch, so torch is deliberately left
alone here. The other pins match the benchmark environment.

Restart the session if pip asks you to, then continue from cell 4.

In [ ]:
%pip install -q --upgrade --upgrade-strategy only-if-needed \
  "gradio>=5.0,<6" "transformers==5.14.1" "bitsandbytes==0.49.2" \
  "accelerate>=1.14" "huggingface_hub>=1.2,<2" "sentencepiece>=0.2,<0.3" \
  "psutil>=5.9" "python-dotenv>=1.0"

import importlib

for name in ("gradio", "transformers", "bitsandbytes", "accelerate", "pandas"):
    try:
        module = importlib.import_module(name)
        print(f"{name:<16} {getattr(module, '__version__', 'unknown')}")
    except Exception as exc:
        print(f"{name:<16} FAILED: {type(exc).__name__}: {exc}")

## 4. Hugging Face token

Read from Colab secrets so the token is never typed into a cell and never
saved in the notebook output.

In [ ]:
import os

from google.colab import userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as exc:
    raise SystemExit(
        "Could not read the Colab secret 'HF_TOKEN'.\n"
        "Open the key icon in the left sidebar, add a secret named HF_TOKEN, "
        "and switch on 'Notebook access'.\n"
        f"Details: {type(exc).__name__}: {exc}"
    )

# Confirm the token works and can see the quantized repositories.
from huggingface_hub import HfApi

api = HfApi(token=os.environ["HF_TOKEN"])
print("Logged in as:", api.whoami()["name"])

REPOS = [
    "google/medgemma-1.5-4b-it",
    "google/gemma-4-E4B-it",
    "google/gemma-4-12B-it",
    "pupupapapa/medgemma-1.5-4b-it-int4-bnb",
    "pupupapapa/medgemma-1.5-4b-it-int8-bnb",
    "pupupapapa/gemma-4-e4b-it-int4-bnb",
    "pupupapapa/gemma-4-e4b-it-int8-bnb",
    "pupupapapa/gemma-4-12b-it-int4-bnb",
    "pupupapapa/gemma-4-12b-it-int8-bnb",
]
print("\nAccess check:")
for repo in REPOS:
    try:
        api.model_info(repo)
        print(f"  ok       {repo}")
    except Exception as exc:
        print(f"  BLOCKED  {repo}  ({type(exc).__name__})")
print("\nA BLOCKED google/* repo means the licence has not been accepted yet.")

## 5. Launch

This prints a `*.gradio.live` public link, valid for 72 hours. The first
question downloads the checkpoint, so it takes a few minutes; later questions
reuse the loaded model.

Stop the cell to shut the app down.

In [ ]:
import sys

WEBAPP = "/content/medpubcodex/webapp"
if WEBAPP not in sys.path:
    sys.path.insert(0, WEBAPP)

import app

app.build_interface().launch(share=True)

## Optional: run the tests

None of these load a model, so they finish in seconds.

In [ ]:
!cd /content/medpubcodex && python -m pytest webapp/tests -q